# Example 9 — Laura++-style veto maps

Invariant-mass and functional vetoes for B+ -> K+ pi+ pi-. The same veto is used in data selection and PDF normalization.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import (CompositeVeto, DecayChannel, DecayModel, FunctionalVeto, MassWindowVeto, NonResonant, RealImag, Resonance, VetoedDensity, enable_x64, vetoed_signal_pdf)
from dalitzplotfitter.background import FunctionalBackground
enable_x64()
channel=DecayChannel('B+',('K+','pi+','pi-'))
components=[Resonance('Kstar892',(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),Resonance('rho770',(1,2),RealImag(0.65,0.10),mass=0.7753,width=0.1491,spin=1),NonResonant(RealImag(-0.5,0.1))]
model=DecayModel(channel,components,normalization_method='square-dalitz',normalization_resolution=300,normalization_pair=(0,2))


## 1. Define vetoes

Mass-window limits are given in GeV, not GeV^2.


In [ ]:
veto_kpi=MassWindowVeto((0,2),1.82,1.91)
veto_pipi=MassWindowVeto((1,2),3.00,3.20)
edge_veto=FunctionalVeto(lambda d:(d['s13']<24.0)&(d['s23']<22.0))
veto=CompositeVeto(veto_kpi,veto_pipi,edge_veto)


## 2. Apply the same veto to an event sample


In [ ]:
raw=model.generate_phase_space(120000,seed=9001)
accepted=veto.apply(raw)
print('raw:',raw.size,'accepted:',accepted.size,'fraction:',accepted.size/raw.size)
fig,axes=plt.subplots(1,2,figsize=(12.5,5),constrained_layout=True)
for ax,sample,title in ((axes[0],raw,'Before veto'),(axes[1],accepted,'After veto')):
    h=ax.hist2d(np.asarray(sample.s13),np.asarray(sample.s23),bins=80)
    fig.colorbar(h[3],ax=ax,label='events')
    ax.set(xlabel=r'$s_{13}$ [GeV$^2$]',ylabel=r'$s_{23}$ [GeV$^2$]',title=title)
plt.show()


## 3. Veto-aware signal normalization

The fitted signal is P = V epsilon |A|^2 / integral(V epsilon |A|^2 dPhi).


In [ ]:
pdf=vetoed_signal_pdf(model,veto)
ordinary=model.pdf()
ordinary_norm=ordinary.normalization({}); vetoed_norm=pdf.normalization({})
print('ordinary normalization:',float(ordinary_norm))
print('vetoed normalization:',float(vetoed_norm))
print('accepted integral ratio:',float(vetoed_norm/ordinary_norm))
probe=raw.as_dict(); mask=np.asarray(veto(probe)); values=np.asarray(pdf(probe,{}))
print('max PDF in vetoed points:',float(values[~mask].max()))


## 4. Apply the same veto to a background shape


In [ ]:
mK,mpi,_=channel.daughter_masses; mB=channel.parent_mass
s13_min,s13_max=(mK+mpi)**2,(mB-mpi)**2
background=FunctionalBackground(lambda d:0.5+1.3*jnp.clip((d['s13']-s13_min)/(s13_max-s13_min),0,1))
vetoed_background=VetoedDensity(background,veto)
norm=model.normalization_sample
bkg_all=jnp.mean(norm.weights*background(norm.as_dict()))
bkg_veto=jnp.mean(norm.weights*vetoed_background(norm.as_dict()))
print('background normalization before veto:',float(bkg_all))
print('background normalization after veto:',float(bkg_veto))


## 5. Visualize the accepted signal PDF


In [ ]:
plot_sample=model.generate_phase_space(100000,seed=9002)
plot_values=np.asarray(pdf(plot_sample.as_dict(),{}))
fig,ax=plt.subplots(figsize=(7,5.5))
h=ax.hist2d(np.asarray(plot_sample.s13),np.asarray(plot_sample.s23),bins=90,weights=plot_values)
fig.colorbar(h[3],ax=ax,label='accepted signal PDF')
ax.set(xlabel=r'$s_{13}$ [GeV$^2$]',ylabel=r'$s_{23}$ [GeV$^2$]',title='Signal PDF with veto maps')
plt.show()
